# 🧮 fxpricer: NDF e opções de USD/BRL

Este notebook mostra o uso do pacote: **edite os parâmetros da célula 3**, rode tudo e veja
preço, Greeks e gráficos.

> ⚠️ Os valores de mercado abaixo são **ilustrativos**, não são cotações reais.
> Projeto de estudo, sem fins de recomendação de investimento.

In [1]:
import sys
from pathlib import Path
from datetime import date

try:
    import fxpricer  # pacote instalado com `pip install -e .`
except ImportError:  # roda direto da pasta do repositório
    sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import replace

from fxpricer import (
    NDF, FXOption, CurvaPre, dias_uteis, dias_corridos,
    payoff_europeia, payoff_asiatico_ndf, payoff_asiatica_opcao, payoff_ndf_com_cap,
    preco_mc, simular_trajetorias,
)
from fxpricer.plots import (
    plot_payoff_ndf, plot_preco_greeks, plot_curva_forward,
    plot_trajetorias, plot_convergencia_mc,
)

DOCS = Path("..") / "docs"
DOCS.mkdir(exist_ok=True)
print("✅ pacote carregado")

✅ pacote carregado


## 🧾 1. Parâmetros da operação (edite aqui)

In [2]:
# 🧾 Operação
spot       = 5.40          # 💵 USD/BRL à vista
strike     = 5.45          # 🎯 taxa contratada
nocional   = 1_000_000     # 💰 em USD
vol        = 0.14          # 📈 volatilidade anual
tipo       = "call"        # 🔁 call ou put

# 📅 Datas (o código calcula dias úteis e corridos)
hoje       = date(2026, 11, 25)
vencimento = date(2027, 2, 24)
feriados   = [date(2026, 12, 25), date(2027, 1, 1)]  # exemplo: complete com o calendário ANBIMA

# 🏦 Mercado (ilustrativo; veja a célula opcional abaixo para dados online)
taxa_pre   = 0.145         # DI x pré efetiva a.a. para o prazo
taxa_usd   = 0.043         # taxa USD simples a.a.

du = dias_uteis(hoje, vencimento, feriados)
dc = dias_corridos(hoje, vencimento)
print(f"Prazo: {du} dias úteis | {dc} dias corridos")

Prazo: 63 dias úteis | 91 dias corridos


### 🌐 (Opcional) Dados online

Troque `USAR_DADOS_ONLINE` para `True` para buscar a PTAX no Banco Central e a SOFR no FRED.
Requer internet. A volatilidade histórica da PTAX serve de aproximação da implícita.

In [3]:
USAR_DADOS_ONLINE = False

if USAR_DADOS_ONLINE:
    from fxpricer.data import baixar_ptax, baixar_sofr, vol_historica
    ptax = baixar_ptax(date(2026, 1, 1), hoje)
    spot = float(ptax.iloc[-1])
    taxa_usd = baixar_sofr()
    vol = vol_historica(ptax, janela=63)
    print(f"PTAX {spot:.4f} | SOFR {taxa_usd:.2%} | vol histórica {vol:.2%}")
else:
    print("Usando parâmetros manuais da célula anterior.")

Usando parâmetros manuais da célula anterior.


## 💵 2. NDF

In [4]:
ndf = NDF(spot=spot, strike=strike, du=du, dc=dc, taxa_pre=taxa_pre,
          taxa_usd=taxa_usd, nocional=nocional, compra=True)

print(f"Forward justo      : {ndf.forward():.4f}")
print(f"Pontos forward     : {ndf.pontos_forward():.4f}")
print(f"MTM (R$)           : {ndf.valor_presente():,.2f}")

fig = plot_payoff_ndf(ndf)
fig.savefig(DOCS / "ndf_payoff.png", dpi=150)
plt.show()

Forward justo      : 5.5259
Pontos forward     : 0.1259
MTM (R$)           : 73,337.30


## 🎯 3. Opção europeia (Garman-Kohlhagen) e Greeks

In [5]:
op = FXOption(spot=spot, strike=strike, tipo=tipo, du=du, dc=dc,
              taxa_pre=taxa_pre, taxa_usd=taxa_usd, vol=vol, nocional=1.0)

tabela = pd.Series(op.greeks(), name=f"{tipo} K={strike}").to_frame()
tabela

,call K=5.45
preco,0.187472
delta,0.585639
gamma,1.017560
vega_1vol,0.010357
theta_1dia,-0.002023
rho_brl_1bp,0.000065
rho_usd_1bp,-0.000079


In [6]:
fig = plot_preco_greeks(op)
fig.savefig(DOCS / "opcao_greeks.png", dpi=150)
plt.show()

## 📈 4. Curva forward

Curva pré com **vértices ilustrativos** e interpolação flat forward. Com dados reais, carregue o
CSV das Taxas Referenciais da B3 com `fxpricer.data.carregar_curva_pre_csv`.

In [7]:
curva = CurvaPre({21: 0.1490, 63: 0.1475, 126: 0.1450, 252: 0.1400, 504: 0.1380})

fig = plot_curva_forward(spot, curva, taxa_usd, du_max=504)
fig.savefig(DOCS / "curva_forward.png", dpi=150)
plt.show()

## 🎲 5. Monte Carlo

Primeiro a **validação**: uma opção europeia por simulação tem de bater com a fórmula fechada.

In [8]:
tempos_1 = np.array([dc / 365])
ns = [2_000, 10_000, 50_000, 250_000, 1_000_000]
resultados = []
for n in ns:
    resultados.append(preco_mc(
        payoff_europeia(strike, tipo), spot, op.forward, vol, tempos_1,
        df_desconto=op.df_brl, n_caminhos=n, barra=False,
    ))

print(f"Fórmula fechada : {op.preco():.6f}")
r = resultados[-1]
print(f"Monte Carlo     : {r['preco']:.6f}  (IC95% {r['ic95_inf']:.6f} a {r['ic95_sup']:.6f})")

fig = plot_convergencia_mc(resultados, referencia=op.preco())
fig.savefig(DOCS / "convergencia_mc.png", dpi=150)
plt.show()

Fórmula fechada : 0.187472
Monte Carlo     : 0.187559  (IC95% 0.187228 a 0.187890)


### Produtos que dependem da trajetória

NDF Asiático (média dos fixings), opção asiática e NDF com cap, com 1 fixing por dia útil.

In [9]:
tempos = np.linspace(dc / 365 / du, dc / 365, du)   # um fixing por dia útil
cap = strike * 1.05

produtos = {
    "NDF Asiático (compra USD)": payoff_asiatico_ndf(strike, nocional=1.0),
    f"Opção asiática {tipo}":    payoff_asiatica_opcao(strike, tipo),
    "NDF com cap (+5%)":         payoff_ndf_com_cap(strike, cap, nocional=1.0),
}

linhas = {}
for nome, p in produtos.items():
    linhas[nome] = preco_mc(p, spot, op.forward, vol, tempos,
                            df_desconto=op.df_brl, n_caminhos=200_000)
pd.DataFrame(linhas).T[["preco", "erro_padrao", "ic95_inf", "ic95_sup"]]

🎲 Monte Carlo:   0%|          | 0/200000 [00:00<?, ?it/s]

🎲 Monte Carlo:  50%|█████     | 100000/200000 [00:00<00:00, 987996.97it/s]

🎲 Monte Carlo: 100%|██████████| 200000/200000 [00:00<00:00, 1002207.61it/s]

🎲 Monte Carlo:   0%|          | 0/200000 [00:00<?, ?it/s]

🎲 Monte Carlo:  60%|██████    | 120000/200000 [00:00<00:00, 1033277.04it/s]

🎲 Monte Carlo: 100%|██████████| 200000/200000 [00:00<00:00, 1040753.44it/s]

🎲 Monte Carlo:   0%|          | 0/200000 [00:00<?, ?it/s]

🎲 Monte Carlo:  60%|██████    | 120000/200000 [00:00<00:00, 1121681.60it/s]

🎲 Monte Carlo: 100%|██████████| 200000/200000 [00:00<00:00, 1130174.61it/s]

,preco,erro_padrao,ic95_inf,ic95_sup
NDF Asiático (compra USD),0.013245,0.000024,0.013197,0.013293
Opção asiática call,0.092881,0.000218,0.092453,0.093309
NDF com cap (+5%),-0.001670,0.000269,-0.002197,-0.001142


In [10]:
caminhos = simular_trajetorias(spot, op.forward, vol, tempos, n_caminhos=500, seed=1)
fig = plot_trajetorias(tempos, caminhos, strike=strike)
fig.savefig(DOCS / "trajetorias.png", dpi=150)
plt.show()

## ⚠️ Limitações

- Volatilidade **constante** (sem smile ou superfície). Na prática, USD/BRL tem smile relevante.
- Calendário simplificado: sábado e domingo, mais os feriados que você informar.
- NDF com cap segue uma **premissa simplificada** (veja a docstring); contratos reais variam.
- Dados de mercado de exemplo são ilustrativos.